#**Connect-Tel Churn Prediction: Optimized XGBoost Model**

##Step 1: Environment and Directory Setup

In this initial step, we import the necessary libraries for data manipulation, machine learning, and performance evaluation. We also ensure that the local directory structure is ready to store our models and outputs.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb

# Creating directories for local organization
os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

##Step 2: Data Loading and Preliminary Cleaning

We load the raw customer data and perform basic data types fixes. This includes handling the total_charges column, which often contains non-numeric strings, and filling any missing values with the median.

In [2]:
print("--- Step 1: Loading and Engineering Features ---")
data_path = '../data/raw/telecom_churn.csv'
try:
    df = pd.read_csv(data_path)
    print(f"Data Loaded Successfully: {df.shape}")
except FileNotFoundError:
    print(f"Error: {data_path} not found. Ensure the CSV is in the correct directory.")
    exit()

# Dropping identifiers and cleaning numeric columns
df.drop(columns=['customer_id'], inplace=True, errors='ignore')
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

--- Step 1: Loading and Engineering Features ---
Data Loaded Successfully: (25000, 36)


##Step 3: Advanced Feature Engineering

In this step, we create Interaction Features and Behavioral Metrics. These "Secret Sauce" features help the XGBoost model identify complex patterns, such as customers experiencing "Bill Shock" or technical frustrations.

In [3]:
# Interaction Features: Combining metrics to find hidden churn drivers
df["tenure_per_charge"] = df["tenure_months"] / (df["monthly_charges"] + 1)
df["technical_friction"] = df["dropped_call_rate"] * (df["num_complaints_3m"] + 1)
df["value_ratio"] = df["arpu"] / (df["monthly_charges"] + 1)

# Behavioral Features
df["complaint_intensity"] = df["num_complaints_3m"] + df["num_complaints_12m"]
df["bill_shock"] = np.where(df["monthly_charges"] > df["monthly_charges"].quantile(0.75), 1, 0)

# Categorical Bucketing
df["tenure_bucket"] = pd.cut(df["tenure_months"],
                             bins=[0, 12, 24, 48, 72, 120],
                             labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr", "6yr+"])

#Step 4: Feature Selection & Preprocessing Pipeline

We use a correlation threshold to filter out "noise" and keep only the most predictive features. Then, we build a ColumnTransformer to automate the scaling of numbers and the encoding of categorical text.

In [4]:
print("--- Step 2: Preparing Preprocessing Pipeline ---")
THRESHOLD = 0.05
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
selected_num_features = corr_matrix['is_churn'][abs(corr_matrix['is_churn']) > THRESHOLD].index.tolist()

if 'is_churn' in selected_num_features:
    selected_num_features.remove('is_churn')

selected_cat_features = ['contract_type', 'connection_type', 'plan_type', 'segment_value', 'tenure_bucket']

X = df[selected_num_features + selected_cat_features]
y = df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), selected_num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), selected_cat_features)
])

--- Step 2: Preparing Preprocessing Pipeline ---


##Step 5: Model Training with Optimized Parameters

We initialize the XGBClassifier using the optimal parameters discovered during hyperparameter tuning. This includes a specific scale_pos_weight to ensure the model focuses on identifying churners (the minority class).

In [5]:
print("--- Step 3: Training Optimized XGBoost Model ---")

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.7,
        scale_pos_weight=1.4137,
        eval_metric='logloss',
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

--- Step 3: Training Optimized XGBoost Model ---


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


##Step 6: Final Evaluation and Decision Policy

We evaluate the model using a custom Decision Threshold of 0.42. This maximizes Recall, allowing the business to capture more than 75% of departing customers.

In [6]:
print("\n--- Step 4: Final Evaluation ---")
y_probs = final_model.predict_proba(X_test)[:, 1]

# Decision Threshold optimized for Recall (0.42)
optimal_threshold = 0.42
y_preds = (y_probs >= optimal_threshold).astype(int)

print("==================================================")
print(f"   FINAL AUC-ROC SCORE: {roc_auc_score(y_test, y_probs):.4f}")
print("==================================================")
print(classification_report(y_test, y_preds))


--- Step 4: Final Evaluation ---
   FINAL AUC-ROC SCORE: 0.6564
              precision    recall  f1-score   support

           0       0.72      0.44      0.55      2929
           1       0.49      0.76      0.60      2071

    accuracy                           0.57      5000
   macro avg       0.61      0.60      0.57      5000
weighted avg       0.63      0.57      0.57      5000



##Step 7: Model Export for Deployment

The final step is to save the complete pipeline (including the preprocessor) as a .pkl file. This file can be loaded later to make predictions on new customer data without re-training.

In [7]:
model_path = "../models/connecttel_churn_pipeline.pkl"
joblib.dump(final_model, model_path)
print(f"--- Step 5: SUCCESS! Model saved to {model_path} ---")

--- Step 5: SUCCESS! Model saved to ../models/connecttel_churn_pipeline.pkl ---
